# Week 13 Lab – Dynamic Programming & Profiling/Optimisation  

## Objectives
- Apply **Dynamic Programming (DP)** to classical problems like **Longest Increasing Subsequence (LIS)** and **Knapsack**.  
- Practice writing **efficient DP solutions** and understand trade-offs in **time and space complexity**.  
- Learn to **profile Python code** using `cProfile` and apply **optimisations** to improve performance.  
- Connect theoretical concepts of DP with **practical profiling and optimisation** workflows used in real-world systems.  


# Q1. Longest Increasing Subsequence (LIS) – From Naïve to Efficient  

The **Longest Increasing Subsequence (LIS)** problem is a classical DP problem with multiple approaches:  
- **Naïve recursive approach** (exponential time).  
- **DP with memoisation/tabulation** (O(n²) time).  
- **Optimised solution with Binary Search + DP** (O(n log n) time).  

### Tasks
1. Implement the **naïve recursive solution** for LIS.  
2. Implement the **DP (tabulation or memoisation) solution**.  
3. Implement the **optimised O(n log n) solution**.  
4. Use **`cProfile`** to profile all three versions on a **large random sequence** (length ~5000). Compare runtimes.  
5. Summarise your observations:  
   - Which solution scales better?  
   - What are the trade-offs in implementation complexity vs runtime?  

In [ ]:
def lis_recursive_helper(seq, prev, cur):
    if cur == len(seq):
        return 0
    taken = 0
    if seq[cur] > prev:
        taken = 1 + lis_recursive_helper(seq, seq[cur], cur + 1)
    not_taken = lis_recursive_helper(seq, prev, cur + 1)
    return max(taken, not_taken)

def lis_recursive(seq):
    return lis_recursive_helper(seq, float("-inf"), 0)

def lis_dp(seq):
    n = len(seq)
    dp = [1] * n
    for i in range(1, n):
        for j in range(i):
            if seq[i] > seq[j]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

import bisect

def lis_optimized(seq):
    sub = []
    for x in seq:
        pos = bisect.bisect_left(sub, x)
        if pos == len(sub):
            sub.append(x)
        else:
            sub[pos] = x
    return len(sub)


# Profiling section
if __name__ == "__main__":
    import cProfile
    import random
    
    seq = [random.randint(1, 10000) for _ in range(2000)]  
    
    print("Profiling LIS Recursive")
    cProfile.run("lis_recursive(seq)")
    
    print("Profiling LIS DP O(n^2)")
    cProfile.run("lis_dp(seq)")
    
    print("Profiling LIS Optimized O(n log n)")
    cProfile.run("lis_optimized(seq)")

# Q2. 0/1 Knapsack – Profiling Space vs Time Trade-offs

The 0/1 Knapsack problem can be solved with:
- Naïve recursion (exponential).
- DP with a 2D table (O(n·W) time, O(n·W) space).
- Optimised DP with 1D array (O(n·W) time, O(W) space).

### Tasks

1. Implement the three approaches for 0/1 Knapsack.
2. For a dataset with n = 200 items and W = 1000 capacity:
    - Run all implementations and profile with cProfile.
    - Record memory consumption observations (hint: use sys.getsizeof() for DP tables).
3. Try to optimise further:
    - Can you prune recursion using memoisation?
    - Can you reduce redundant computations in the iterative DP?
4. Summarise your observations:
    - How does space optimisation affect runtime?
    - Which approach is more practical in large-scale systems?

In [ ]:
def knapsack_recursive(weights, values, capacity, n):
    if n == 0 or capacity == 0:
        return 0
    if weights[n-1] > capacity:
        return knapsack_recursive(weights, values, capacity, n-1)
    else:
        return max(values[n-1] + knapsack_recursive(weights, values, capacity-weights[n-1], n-1),
                   knapsack_recursive(weights, values, capacity, n-1))

def knapsack_dp(weights, values, capacity, n):
    dp = [[0 for _ in range(capacity+1)] for _ in range(n+1)]
    for i in range(1, n+1):
        for w in range(1, capacity+1):
            if weights[i-1] <= w:
                dp[i][w] = max(values[i-1] + dp[i-1][w-weights[i-1]], dp[i-1][w])
            else:
                dp[i][w] = dp[i-1][w]
    return dp[n][capacity]

def knapsack_space_optimized(weights, values, capacity, n):
    dp = [0] * (capacity+1)
    for i in range(n):
        for w in range(capacity, weights[i]-1, -1):
            dp[w] = max(dp[w], values[i] + dp[w-weights[i]])
    return dp[capacity]


# Profiling section
if __name__ == "__main__":
    import cProfile
    import random
    
    n = 200
    capacity = 1000
    weights = [random.randint(1, 50) for _ in range(n)]
    values = [random.randint(10, 100) for _ in range(n)]
    
    print("Profiling Knapsack Recursive (Warning: very slow)")
    cProfile.run("knapsack_recursive(weights, values, capacity, n)")
    
    print("Profiling Knapsack DP (2D)")
    cProfile.run("knapsack_dp(weights, values, capacity, n)")
    
    print("Profiling Knapsack Space Optimized (1D)")
    cProfile.run("knapsack_space_optimized(weights, values, capacity, n)")

# Q3. Case Study – Optimising a Real-World Pipeline with DP + Profiling

Imagine you are tasked with designing a budget allocation system for a startup.
- There are multiple projects, each requiring some budget (like weight) and yielding some ROI (Return on Investment) (like value).
- The company has a fixed budget (capacity).
- The goal is to maximise ROI while staying within budget (a knapsack variant).

Now, consider that:
- Each project may have dependencies (e.g., Project B requires Project A to be funded first).
- You must ensure the solution respects dependencies.

### Tasks

1. Design a DP-based solution that handles knapsack with dependencies.
    - Hint: Use graph/topological sorting + knapsack.
2. Profile the solution using cProfile on large datasets (n ~ 500 projects).
3. Optimise your approach and compare profiling results.
    - Try restructuring data, memoisation, or iterative vs recursive design.
4. Write a report cell in Markdown:
    - Summarise bottlenecks discovered via profiling.
    - Discuss how optimisations improved performance.
    - Reflect on scalability: Would your approach work for n = 10^5 projects? Why or why not?

In [ ]:
from collections import defaultdict, deque

def topological_sort(n, dependencies):
    indeg = [0] * n
    graph = defaultdict(list)
    for u, deps in dependencies.items():
        for v in deps:
            graph[v].append(u)
            indeg[u] += 1
    q = deque([i for i in range(n) if indeg[i] == 0])
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order

def knapsack_with_dependencies(projects, budget):
    """
    projects: list of tuples (id, cost, roi, dependencies)
    budget: total budget
    """
    n = len(projects)
    dependencies = {pid: deps for pid, cost, roi, deps in projects}
    
    order = topological_sort(n, dependencies)
    
    dp = [0] * (budget+1)
    costs = {pid: cost for pid, cost, roi, deps in projects}
    rois = {pid: roi for pid, cost, roi, deps in projects}
    
    for pid in order:
        cost = costs[pid]
        roi = rois[pid]
        for b in range(budget, cost-1, -1):
            dp[b] = max(dp[b], roi + dp[b-cost])
    return dp[budget]


# Profiling section
if __name__ == "__main__":
    import cProfile
    import random
    
    n = 500
    projects = []
    for i in range(n):
        cost = random.randint(1, 50)
        roi = random.randint(10, 100)
        dependencies = [random.randint(0, i-1)] if i > 0 and random.random() < 0.2 else []
        projects.append((i, cost, roi, dependencies))
    
    budget = 5000
    
    print("Profiling Knapsack with Dependencies")
    cProfile.run("knapsack_with_dependencies(projects, budget)")